<a href="https://colab.research.google.com/github/cpdong/public/blob/master/test/LLMBind_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **LLMBind**

Generate de novo protein binder sequences from a target PDB structure.

This Colab contains only two modules:

1. **Setup LLMBind (~3 min)**
2. **Run LLMBind to generate binders**


In [ ]:
#@title (1) setup LLMBind (~3 min)
%%time
%%bash
set -e

# working directory
WORK_DIR="/content/LLMBind_demo"
mkdir -p "$WORK_DIR"
cd "$WORK_DIR"

# Python dependencies for Colab TPU setup
pip install --upgrade tensorflow
pip install accelerate silence_tensorflow==1.2.3 tensorflow==2.20.0 tensorflow_cpu==2.20.0

# JAX TPU build. For TPU, use libtpu release index instead of CUDA release index.
pip install -U "jax[tpu]==0.5.3" flax dm-haiku -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

pip install git+https://github.com/cpdong/NetSurfP_3.0_standalone.git
pip install numpy==1.26.4 pyyaml==6.0.2 requests==2.32.3 fair-esm scikit-learn==1.6.1 scipy==1.13.1
pip install transformers==4.57.3 torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 Bio typing

# preload ESM model data
python -c "import esm; esm.pretrained.esm1b_t33_650M_UR50S()" # preload data for NetSurfP_3.0
python -c "import esm; esm.pretrained.esm2_t6_8M_UR50D()" # preload data for bindscan

# download run script
wget -q https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test/run_test.py -O run_test.py
wget -q https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test/bs_model.pt -O bs_model.pt
wget -q https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test/PDL1.fasta -O PDL1.fasta
wget -q https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test/PDL1.pdb -O PDL1.pdb
echo "LLMBind setup completed."


In [ ]:
#@title (2) run LLMBind to generate binders
%%time

import os, shlex, subprocess
from pathlib import Path
from google.colab import files

# ---- input target PDB ----
upload_target_pdb = True #@param {type:"boolean"}
target_pdb = "/content/target.pdb" #@param {type:"string"}

if upload_target_pdb:
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("No PDB file uploaded.")
    target_pdb = "/content/" + list(uploaded.keys())[0]

# ---- model paths ----
llm_model = "/content/llm_model" #@param {type:"string"}
ppi_model = "/content/ppi_model" #@param {type:"string"}

# ---- design options ----
num_designs = 100 #@param {type:"integer"}
generate_batch_size = 16 #@param {type:"integer"}
max_length = 130 #@param {type:"integer"}
min_structured_fraction = 0.6 #@param {type:"number"}
output_dir = "/content/llmbind_output" #@param {type:"string"}

Path(output_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    "python", "/content/LLMBind_demo/run_test.py",
    "--target_pdb", target_pdb,
    "--gen_model", llm_model,
    "--generate_batch_size", str(generate_batch_size),
    "--max_length", str(max_length),
    "--min_structured_fraction", str(min_structured_fraction),
    "--output_dir", output_dir,
    "--num_designs", str(num_designs),
    "--ppi_model", ppi_model,
]

print("Running command:
" + " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

print("
Finished. Output files:")
for path in sorted(Path(output_dir).rglob("*")):
    if path.is_file():
        print(path)
